In [1]:
from utils.data import ProteinDataset, ProteinPairDataset
import torch as pt
import numpy as np


data = pt.load(f'./data/pbond0_hbond0.pt', weights_only=False)
datalib = ProteinDataset(data)
lib_map = np.arange(len(datalib), dtype=np.int64)
print('number of proteins:', len(lib_map))
pdb2idx = [(data[2][i], i) for i in range(len(data[2]))] # pdb name -> idx
pdb2idx = dict(pdb2idx)
pair_dataset = ProteinPairDataset(datalib, './data/tmalign.out', pdb2idx)

number of proteins: 106973


In [2]:
import torch.nn as nn
import torch_geometric.nn as gnn
from torch.nn import TransformerEncoder, TransformerEncoderLayer


class ProteinGCN(nn.Module):
    def __init__(self, embed_dim:int=512, hidden_channels:int=256, out_channels:int=128, num_layers:int=3, num_edge_features:int=10,):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings=21, embedding_dim=embed_dim, padding_idx=0,)
        # node_attr占一维
        self.gcn = gnn.GCN(in_channels=embed_dim+num_edge_features, hidden_channels=hidden_channels, 
                           num_layers=num_layers, out_channels=out_channels,)
        self.shared = nn.Sequential(nn.Linear(4*out_channels, out_channels), nn.SiLU(),)
        self.tm_head = nn.Linear(out_channels, 1,)
        self.seq_head = nn.Linear(out_channels, 1,)
        # encoder_layer = TransformerEncoderLayer(d_model=embed_dim, nhead=8, dim_feedforward=embed_dim*4, 
                                                # activation='gelu', batch_first=True,)
        # self.encoder = TransformerEncoder(encoder_layer, num_layers=1)
        # self.mlp = nn.Linear(embed_dim, 21)

    def embed(self, seq_mask, mode:str='train'):
        seq, mask = seq_mask
        embedding = self.emb(seq) # [batch_size, seq_len, emb_dim]
        embedding = embedding * mask.unsqueeze(-1) # mask: [batch_size, seq_len, 1]
        if mode == 'emb':
            embedding = embedding.sum(dim=1) # [batch_size, emb_dim]
        return embedding

    def encode_protein(self, seq, mask, graph):
        x, edge_idx, edge_attr, batch, node2seq = graph.x, graph.edge_index, graph.edge_attr, graph.batch, graph.node2seq
        emb = self.embed((seq, mask))
        B, L, D = emb.shape
        emb_flat = emb.view(-1, D)
        flat_idx = batch * L + node2seq
        node_emb = emb_flat[flat_idx]
        x = pt.cat([node_emb, x], dim=-1)
        x = self.gcn(x, edge_idx, edge_attr=edge_attr, batch=batch)
        x = gnn.global_mean_pool(x, batch)
        return x        

    def forward(self, data, mode:str='finetuning'):
        if mode == 'pretraining':
            seqs_pad, masks_pad = data
            embs = self.emb(seqs_pad) # [batch_size, seq_len, emb_dim]
            embs = self.encoder(embs, src_key_padding_mask=~masks_pad) # [batch_size, seq_len, emb_dim]
            outputs = self.mlp(embs) # [batch_size, seq_len, 21]
            return outputs
        elif mode == 'finetuning':            
            (seqs, masks, graphs), (inv_i, inv_j) = data
            prot_repr = self.encode_protein(seqs, masks, graphs)
            x_i = prot_repr[inv_i]
            x_j = prot_repr[inv_j]
            feature = pt.cat([x_i, x_j, x_i-x_j, x_i*x_j], dim=-1)
            shared = self.shared(feature)
            tm_score = self.tm_head(shared).squeeze(-1)
            seq_score = self.seq_head(shared).squeeze(-1)
            return tm_score, seq_score
        else:
            raise ValueError(f'Unknown mode: {mode}')

In [5]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from utils.data import pair_collate_fun, collate_fun_emb




libloader = DataLoader(datalib, batch_size=512, shuffle=False, collate_fn=collate_fun_emb, num_workers=6)
pair_map = np.arange(len(pair_dataset), dtype=np.int64)
print('len pair_dataset:', len(pair_map))
_, test_map = train_test_split(lib_map, test_size=1024, random_state=42)
train_set = ProteinPairDataset(pair_dataset, mapping=pair_map)
test_set = ProteinDataset(datalib, mapping=test_map)
train_loader = DataLoader(train_set, batch_size=384, shuffle=True, 
                          collate_fn=pair_collate_fun(datalib), drop_last=True, num_workers=6)
test_loader = DataLoader(test_set, batch_size=512, shuffle=False, 
                         collate_fn=collate_fun_emb, num_workers=6)

len pair_dataset: 1214640


In [ ]:
from tqdm import tqdm
from utils.tools import gen_embeddings, build_idx, calculate_score, save_model
from dataclasses import dataclass
import os
import subprocess
import re
import multiprocessing as mp


def run_tmalign(pdb_pair, tmalign_path="./TMalign", reference=1):
    pdb1, pdb2 = pdb_pair
    cmd = [tmalign_path, pdb1, pdb2]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    output = result.stdout
    # aligned_len = int(re.search(r"Aligned length=\s*(\d+)", output).group(1))
    # rmsd = float(re.search(r"RMSD=\s*([0-9.]+)", output).group(1))
    seqid = float(re.search(r"Seq_ID=.*?=\s*([0-9.]+)", output).group(1))
    tm_scores = re.findall(r"TM-score=\s*([0-9.]+)", output)
    tm_score = float(tm_scores[reference - 1])
    score = tm_score - 0.6 + min(0.4 - seqid, 0.0)
    return score 


def generate_tasks(query, database, idx:list, k:int=12):
    for i in range(len(query)):
        q_name = query[i][-1]
        q_file = f'../../data/pdb/{q_name[1:3]}/{q_name}.pdb'
        for j in idx[i][:k]:
            c_name = database[j][-1]
            c_file = f'../../data/pdb/{c_name[1:3]}/{c_name}.pdb'
            yield q_file, c_file


def calculate_score(query, database, idx, k:int=12):
    N = len(query)
    assert N == len(idx), 'The length of query_mols and idx should be the same.'
    with mp.Pool(os.cpu_count() // 2) as p:
        score = p.map(run_tmalign, generate_tasks(query, database, idx, k))
    total = sum(score)
    return total / (k * N)


@dataclass
class TrainConfig:
    epochs: int = 10
    gpu: int = 6
    model_name: str = 'gcn'
    lr: float = 1e-3


class RegressionTrainer:
    def __init__(self, model, config):
        super().__init__()
        self.model = model
        self.config = config
        self.f = open(f'{config.model_name}.txt', 'w')
        self.max_score = 0.0
        self.criterion = nn.SmoothL1Loss()
        self.optimizer = pt.optim.AdamW(self.model.parameters(), lr=config.lr)
        self.scheduler = pt.optim.lr_scheduler.StepLR(self.optimizer, step_size=1, gamma=0.1)

    def train(self, train_loader:DataLoader, test_loader:DataLoader, lib_loader:DataLoader):
        for epoch in range(self.config.epochs):
            print(f'=======================Train_epoch{epoch+1}===========================')
            self.f.write('\nEpoch%d  Training\n' % (epoch+1))
            train_loss = []
            self.model.train()
            for i, batch in enumerate(tqdm(train_loader, unit='batch')):
                prot, inv, score = batch
                prot = [x.to(self.config.gpu) for x in prot]
                inv = [x.to(self.config.gpu) for x in inv]
                score = score.to(self.config.gpu)
                output = self.model((prot, inv))
                tm_score, seq_score = output
                tm_loss = self.criterion(tm_score, score[:, 0])
                seq_loss = self.criterion(seq_score, score[:, 1])
                loss = tm_loss + seq_loss
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                train_loss.append(loss.item())
                if (i+1) % 500 == 0:
                    l = pt.tensor(train_loss).mean()
                    print(f'Epoch [{epoch+1}/{self.config.epochs}], Train Loss: {l:.4f}')
                    self.f.write(f'Epoch [{epoch+1}/{self.config.epochs}], Train Loss: {l:.4f}\n')
                    train_loss = []
            self.scheduler.step()
            print(f'=======================Test_epoch{epoch+1}===========================')
            self.f.write('\nEpoch%d  Testing\n' % (epoch+1))
            pt.cuda.empty_cache()
            embs_lib = gen_embeddings(self.model, lib_loader, self.config.gpu)
            embs_test = gen_embeddings(self.model, test_loader, self.config.gpu)
            I, _ = build_idx(embs_lib, embs_test, self.config.gpu)
            score = calculate_score(test_set, datalib, I)
            print(f'Remote homologous score: {score}')
            self.f.write(f'Remote homologous score: {score}')
            if score > self.max_score:
                self.max_score = score
                save_model(self.model, self.config.model_name)
            print(f'======================================================================')
        self.f.close()   


if __name__ == '__main__':
    config = TrainConfig()
    pt.cuda.set_device(config.gpu)
    model = ProteinGCN().cuda(config.gpu)
    trainer = RegressionTrainer(model, config)
    trainer.train(train_loader, test_loader, libloader)
    pt.cuda.empty_cache()

In [5]:
pt.cuda.empty_cache()

In [ ]:
from tqdm import tqdm
from utils.tools import gen_embeddings, build_idx, save_model
from dataclasses import dataclass
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import subprocess
import re

import torch as pt
import torch.nn as nn
from torch.utils.data import DataLoader


def make_pdb_path(pdb_root, pdb_name: str) -> Path:
    """
    根据 pdb_name 构造 PDB 文件路径。
    例如 1abc -> ../../data/pdb/ab/1abc.pdb
    """
    return Path(pdb_root) / pdb_name[1:3] / f"{pdb_name}.pdb"


def run_tmalign(task):
    pdb1, pdb2, tmalign_path, reference = task
    cmd = [str(tmalign_path), str(pdb1), str(pdb2)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"TMalign failed.\n"
            f"cmd: {' '.join(cmd)}\n"
            f"returncode: {result.returncode}\n"
            f"stderr: {result.stderr}"
        )
    output = result.stdout
    seqid_match = re.search(r"Seq_ID=.*?=\s*([0-9.]+)", output)
    tm_scores = re.findall(r"TM-score=\s*([0-9.]+)", output)
    if seqid_match is None:
        raise ValueError(
            f"Failed to parse Seq_ID from TMalign output.\n"
            f"cmd: {' '.join(cmd)}\n"
            f"stdout:\n{output}"
        )
    if not (1 <= reference <= len(tm_scores)):
        raise ValueError(
            f"Invalid reference={reference}. "
            f"Parsed {len(tm_scores)} TM-scores.\n"
            f"cmd: {' '.join(cmd)}\n"
            f"stdout:\n{output}"
        )
    seqid = float(seqid_match.group(1))
    tm_score = float(tm_scores[reference - 1])
    score = tm_score - 0.6 + min(0.4 - seqid, 0.0)
    return score


def generate_tasks(
    query,
    database,
    idx,
    k: int = 12,
    pdb_root: str = "../../data/pdb",
    tmalign_path: str = "./TMalign",
    reference: int = 1,
):
    """
    生成所有 (q_file, c_file, tmalign_path, reference) 任务
    """
    pdb_root = Path(pdb_root)
    tmalign_path = Path(tmalign_path).resolve()

    if not tmalign_path.is_file():
        raise FileNotFoundError(f"TMalign binary not found: {tmalign_path}")

    for i in range(len(query)):
        q_name = query[i][-1]
        q_file = make_pdb_path(pdb_root, q_name)

        if not q_file.is_file():
            raise FileNotFoundError(f"Query PDB file not found: {q_file}")

        for j in idx[i][:k]:
            c_name = database[int(j)][-1]
            c_file = make_pdb_path(pdb_root, c_name)

            if not c_file.is_file():
                raise FileNotFoundError(f"Candidate PDB file not found: {c_file}")

            yield (q_file, c_file, tmalign_path, reference)


def calculate_remote_homology_score(
    query,
    database,
    idx,
    k: int = 12,
    pdb_root: str = "../../data/pdb",
    tmalign_path: str = "./TMalign",
    reference: int = 1,
    num_workers: int = None,
):
    """
    并行计算 remote homology score

    这里使用 ThreadPoolExecutor，而不是 multiprocessing.Pool：
    - 真正耗时的是外部 TMalign 进程
    - 线程只负责并发调度 subprocess.run
    - 避免在已初始化 CUDA 的训练进程里再 fork 多进程
    """
    N = len(query)
    assert N == len(idx), "The length of query and idx should be the same."

    tasks = list(
        generate_tasks(
            query=query,
            database=database,
            idx=idx,
            k=k,
            pdb_root=pdb_root,
            tmalign_path=tmalign_path,
            reference=reference,
        )
    )

    if len(tasks) == 0:
        raise ValueError("No TM-align tasks were generated.")

    if num_workers is None:
        cpu_count = os.cpu_count() or 1
        num_workers = max(1, min(len(tasks), cpu_count // 2 if cpu_count > 1 else 1))

    total_score = 0.0
    success_cnt = 0
    error_messages = []

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(run_tmalign, task) for task in tasks]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="TM-align",
            leave=False,
        ):
            try:
                score = future.result()
                total_score += score
                success_cnt += 1
            except Exception as e:
                error_messages.append(str(e))

    if success_cnt == 0:
        raise RuntimeError(
            "All TM-align tasks failed.\n"
            + ("\n\nFirst error:\n" + error_messages[0] if error_messages else "")
        )

    if error_messages:
        print(f"[Warning] {len(error_messages)} TM-align tasks failed.")
        print(f"[Warning] First error:\n{error_messages[0]}")

    return total_score / success_cnt


@dataclass
class TrainConfig:
    epochs: int = 10
    gpu: int = 6
    model_name: str = "gcn"
    lr: float = 1e-3
    pdb_root: str = "../../data/pdb"
    tmalign_path: str = "./TMalign"
    topk: int = 12
    tmalign_reference: int = 1
    tmalign_workers: int = None  # None 表示自动选择


class RegressionTrainer:
    def __init__(self, model, config: TrainConfig):
        super().__init__()
        self.model = model
        self.config = config
        self.log_file = open(f"{config.model_name}.txt", "w", encoding="utf-8")
        self.max_score = float("-inf")
        self.criterion = nn.SmoothL1Loss()
        self.optimizer = pt.optim.AdamW(self.model.parameters(), lr=config.lr)
        self.scheduler = pt.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=1, gamma=0.1
        )

    def close(self):
        if not self.log_file.closed:
            self.log_file.close()

    def train(
        self,
        train_loader: DataLoader,
        test_loader: DataLoader,
        lib_loader: DataLoader,
        test_set,
        datalib,
    ):
        try:
            for epoch in range(self.config.epochs):
                print(f"======================= Train epoch {epoch + 1} =======================")
                self.log_file.write(f"\nEpoch {epoch + 1} Training\n")
                train_loss = []
                self.model.train()
                for i, batch in enumerate(tqdm(train_loader, unit="batch")):
                    prot, inv, score = batch
                    prot = [x.to(self.config.gpu) for x in prot]
                    inv = [x.to(self.config.gpu) for x in inv]
                    score = score.to(self.config.gpu)
                    output = self.model((prot, inv))
                    tm_score, seq_score = output
                    tm_loss = self.criterion(tm_score, score[:, 0])
                    seq_loss = self.criterion(seq_score, score[:, 1])
                    loss = tm_loss + seq_loss
                    self.optimizer.zero_grad()
                    loss.backward()
                    self.optimizer.step()
                    train_loss.append(loss.item())
                    if (i + 1) % 500 == 0:
                        avg_loss = float(pt.tensor(train_loss).mean())
                        msg = f"Epoch [{epoch + 1}/{self.config.epochs}], Train Loss: {avg_loss:.4f}"
                        print(msg)
                        self.log_file.write(msg + "\n")
                        train_loss = []

                self.scheduler.step()
                print(f"======================= Test epoch {epoch + 1} =======================")
                self.log_file.write(f"\nEpoch {epoch + 1} Testing\n")
                pt.cuda.empty_cache()
                self.model.eval()
                with pt.no_grad():
                    embs_lib = gen_embeddings(self.model, lib_loader, self.config.gpu)
                    embs_test = gen_embeddings(self.model, test_loader, self.config.gpu)
                I, _ = build_idx(embs_lib, embs_test, self.config.gpu)
                score = calculate_remote_homology_score(
                    query=test_set,
                    database=datalib,
                    idx=I,
                    k=self.config.topk,
                    pdb_root=self.config.pdb_root,
                    tmalign_path=self.config.tmalign_path,
                    reference=self.config.tmalign_reference,
                    num_workers=self.config.tmalign_workers,
                )
                msg = f"Remote homologous score: {score:.6f}"
                print(msg)
                self.log_file.write(msg + "\n")
                if score > self.max_score:
                    self.max_score = score
                    save_model(self.model, self.config.model_name, epoch)
                print("======================================================================")
        finally:
            self.close()


if __name__ == "__main__":
    # 下面这些对象需要你自己在别处定义或导入：
    # ProteinGCN, train_loader, test_loader, lib_loader, test_set, datalib
    config = TrainConfig()
    pt.cuda.set_device(config.gpu)
    model = ProteinGCN().cuda(config.gpu)
    trainer = RegressionTrainer(model, config)
    trainer.train(
        train_loader=train_loader,
        test_loader=test_loader,
        lib_loader=libloader,
        test_set=test_set,
        datalib=datalib,
    )
    pt.cuda.empty_cache()

======================= Train epoch 1 =======================


 16%|█▌        | 501/3163 [00:51<04:17, 10.35batch/s]

Epoch [1/10], Train Loss: 0.0068


 32%|███▏      | 1002/3163 [01:41<03:33, 10.13batch/s]

Epoch [1/10], Train Loss: 0.0048


...

100%|██████████| 3163/3163 [05:18<00:00,  9.92batch/s]

======================= Test epoch 1 =======================


Searching time:  0:00:00.027759


Remote homologous score: -0.231389


TypeError: save_model() missing 1 required positional argument: 'epoch'